# MLOps - DenseNet-121 Training

In [ ]:
import json
from datetime import datetime
from pathlib import Path

import dagshub
import mlflow

import os
from dagshub.auth import add_app_token
from kaggle_secrets import UserSecretsClient

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau,
)
from tensorflow.keras.preprocessing import image_dataset_from_directory

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# BASE_DIR explicite — fonctionne en local ET sur Kaggle/Colab
BASE_DIR = Path(os.environ.get("PROJECT_BASE_DIR", Path.cwd()))

UPLOADS_DIR = BASE_DIR / "uploads"
DATASETS_DIR = BASE_DIR / "uploads" / "datasets"
MODELS_DIR = BASE_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

USER_NAME = "ML_Goat"
REPO_NAME = "Citrus_Disease_MLOps"

# Authentification non-interactive (obligatoire en headless)
def get_dagshub_token():
    # Fonctionne si le notebook est exécuté interactivement dans l'éditeur Kaggle
    try:
        return UserSecretsClient().get_secret("DAGSHUB_TOKEN")
    except Exception:
        pass

    # Fallback pour une exécution déclenchée via l'API (kaggle kernels push)
    token_file = Path("/kaggle/input/citrus-secrets/dagshub_token.txt")
    if token_file.exists():
        return token_file.read_text().strip()

    raise RuntimeError("Impossible de récupérer le token DagsHub (ni Secrets, ni dataset privé).")

dagshub_token = get_dagshub_token()
add_app_token(token=dagshub_token)

# Connecter mlflow avec DagsHub
dagshub.init(repo_owner=USER_NAME, repo_name=REPO_NAME, mlflow=True)

MLFLOW_EXPERIMENT_NAME = "Citrus Disease - DenseNet121 Training"
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

In [ ]:
# ============================================================
# PARAMÈTRES
# ============================================================

IMAGE_SIZE = (224, 224)

BATCH_SIZE = 32

EPOCHS = 50

LEARNING_RATE = 0.0001

SEED = 42

In [ ]:
# ============================================================
# CLASSES
# ============================================================

CLASS_NAMES = [
    "black_spot",
    "canker",
    "greening",
    "healthy",
    "rot",
    "scab",
]

In [ ]:
# ============================================================
# POURCENTAGES
# ============================================================

TRAIN_SIZE = 0.70

VALIDATION_SIZE = 0.15

TEST_SIZE = 0.15

In [ ]:
# ============================================================
# VÉRIFICATION DES POURCENTAGES
# ============================================================

if (
    TRAIN_SIZE
    + VALIDATION_SIZE
    + TEST_SIZE
    != 1.0
):
    raise ValueError(
        "Les proportions train/validation/test "
        "doivent être égales à 1.0"
    )

In [ ]:
# ============================================================
# TROUVER LE DERNIER DATASET
# ============================================================

def get_latest_dataset():

    if not DATASETS_DIR.exists():

        raise FileNotFoundError(
            f"Le dossier {DATASETS_DIR} "
            f"n'existe pas."
        )

    versions = []

    for directory in DATASETS_DIR.iterdir():

        if not directory.is_dir():
            continue

        if not directory.name.startswith("v"):
            continue

        try:

            version_number = int(
                directory.name[1:]
            )

            versions.append(
                (version_number, directory)
            )

        except ValueError:
            continue

    if not versions:

        raise FileNotFoundError(
            "Aucune version de dataset trouvée."
        )

    versions.sort(
        key=lambda x: x[0]
    )

    latest_version, latest_path = versions[-1]

    return (
        f"v{latest_version}",
        latest_path,
    )

In [ ]:
# ============================================================
# VÉRIFICATION DU DATASET
# ============================================================

def verify_dataset(dataset_dir: Path):

    print()
    print("=" * 60)
    print("Vérification du dataset")
    print("=" * 60)

    total_images = 0

    for class_name in CLASS_NAMES:

        class_dir = dataset_dir / class_name

        if not class_dir.exists():

            raise FileNotFoundError(
                f"Classe manquante : "
                f"{class_dir}"
            )

        image_files = [
            file
            for file in class_dir.iterdir()
            if file.is_file()
        ]

        image_count = len(image_files)

        total_images += image_count

        print(
            f"{class_name:15} : "
            f"{image_count} images"
        )

    print("-" * 60)

    print(
        f"{'TOTAL':15} : "
        f"{total_images} images"
    )

    print()

    if total_images < 10:

        raise ValueError(
            "Le dataset contient trop peu "
            "d'images pour créer des splits "
            "train/validation/test."
        )

    return total_images

In [ ]:
# ============================================================
# CRÉATION DES 3 DATASETS
# ============================================================

def create_datasets(dataset_dir: Path):

    print(
        "Création des datasets "
        "train / validation / test..."
    )

    # --------------------------------------------------------
    # Première séparation :
    #
    # 70% TRAIN
    # 30% TEMPORAIRE
    #
    # --------------------------------------------------------

    train_dataset = (
        image_dataset_from_directory(
            dataset_dir,
            labels="inferred",
            label_mode="int",
            class_names=CLASS_NAMES,
            image_size=IMAGE_SIZE,
            batch_size=BATCH_SIZE,
            validation_split=0.30,
            subset="training",
            seed=SEED,
            shuffle=True,
        )
    )

    temporary_dataset = (
        image_dataset_from_directory(
            dataset_dir,
            labels="inferred",
            label_mode="int",
            class_names=CLASS_NAMES,
            image_size=IMAGE_SIZE,
            batch_size=BATCH_SIZE,
            validation_split=0.30,
            subset="validation",
            seed=SEED,
            shuffle=True,
        )
    )

    # --------------------------------------------------------
    # Le temporary contient 30%.
    #
    # On veut :
    #
    # 15% validation
    # 15% test
    #
    # Donc on divise temporaire en deux.
    # --------------------------------------------------------

    temporary_batches = (
        tf.data.experimental.cardinality(
            temporary_dataset
        ).numpy()
    )

    if temporary_batches <= 1:

        raise ValueError(
            "Le dataset temporaire est trop petit "
            "pour créer validation et test."
        )

    validation_batches = (
        temporary_batches // 2
    )

    validation_dataset = (
        temporary_dataset.take(
            validation_batches
        )
    )

    test_dataset = (
        temporary_dataset.skip(
            validation_batches
        )
    )

    return (
        train_dataset,
        validation_dataset,
        test_dataset,
    )

In [ ]:
# ============================================================
# OPTIMISATION DES DATASETS
# ============================================================

def optimize_datasets(
    train_dataset,
    validation_dataset,
    test_dataset,
):

    autotune = tf.data.AUTOTUNE

    train_dataset = (
        train_dataset.prefetch(autotune)
    )

    validation_dataset = (
        validation_dataset.prefetch(autotune)
    )

    test_dataset = (
        test_dataset.prefetch(autotune)
    )

    return (
        train_dataset,
        validation_dataset,
        test_dataset,
    )

In [ ]:
# ============================================================
# CRÉATION DU MODÈLE DENSENET-121
# ============================================================

def create_model():

    print()
    print(
        "Création du modèle DenseNet-121..."
    )

    base_model = DenseNet121(
        weights="imagenet",
        include_top=False,
        input_shape=(
            IMAGE_SIZE[0],
            IMAGE_SIZE[1],
            3,
        ),
    )

    # --------------------------------------------------------
    # Transfer Learning
    # --------------------------------------------------------

    base_model.trainable = False

    # --------------------------------------------------------
    # Input
    # --------------------------------------------------------

    inputs = keras.Input(
        shape=(
            IMAGE_SIZE[0],
            IMAGE_SIZE[1],
            3,
        )
    )

    # --------------------------------------------------------
    # Data Augmentation
    # --------------------------------------------------------

    x = layers.RandomFlip(
        "horizontal"
    )(inputs)

    x = layers.RandomRotation(
        0.1
    )(x)

    x = layers.RandomZoom(
        0.1
    )(x)

    # --------------------------------------------------------
    # DenseNet-121
    # --------------------------------------------------------

    x = base_model(
        x,
        training=False,
    )

    # --------------------------------------------------------
    # Classification Head
    # --------------------------------------------------------

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dropout(
        0.3
    )(x)

    outputs = layers.Dense(
        len(CLASS_NAMES),
        activation="softmax",
    )(x)

    model = keras.Model(
        inputs,
        outputs,
    )

    # --------------------------------------------------------
    # Compilation
    # --------------------------------------------------------

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),
        loss=(
            "sparse_categorical_crossentropy"
        ),
        metrics=[
            "accuracy"
        ],
    )

    return model

In [ ]:
# ============================================================
# CALLBACK MLFLOW
# ============================================================

class MLflowLoggingCallback(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        for key, value in (logs or {}).items():
            if value is not None:
                try:
                    mlflow.log_metric(key, float(value), step=epoch)
                except (TypeError, ValueError):
                    pass

In [ ]:
# ============================================================
# ENTRAÎNEMENT
# ============================================================

def train_model(
    model,
    train_dataset,
    validation_dataset,
    model_path: Path,
):

    print()
    print("=" * 60)
    print("Début de l'entraînement")
    print("=" * 60)

    callbacks = [

        # ----------------------------------------------------
        # Arrête l'entraînement si la validation
        # n'améliore plus
        # ----------------------------------------------------

        EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True,
            verbose=1,
        ),

        # ----------------------------------------------------
        # Sauvegarde le meilleur modèle
        # ----------------------------------------------------

        ModelCheckpoint(
            filepath=model_path,
            monitor="val_accuracy",
            save_best_only=True,
            verbose=1,
        ),

        # ----------------------------------------------------
        # Réduit le learning rate si nécessaire
        # ----------------------------------------------------

        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.2,
            patience=2,
            min_lr=1e-7,
            verbose=1,
        ),
        MLflowLoggingCallback(),
    ]

    history = model.fit(
        train_dataset,
        validation_data=validation_dataset,
        epochs=EPOCHS,
        callbacks=callbacks,
    )

    return history

In [ ]:
# ============================================================
# COMPTER LES IMAGES DANS UN DATASET
# ============================================================

def count_dataset_images(dataset):

    count = 0

    for _, labels in dataset:

        count += labels.shape[0]

    return count

In [ ]:
# ============================================================
# SAUVEGARDER LES MÉTADONNÉES
# ============================================================

def save_training_metadata(
    dataset_version,
    history,
    model_path,
    train_count,
    validation_count,
    test_count,
):

    history_data = {
        key: [
            float(value)
            for value in values
        ]
        for key, values
        in history.history.items()
    }

    best_val_accuracy = max(
        history_data.get(
            "val_accuracy",
            [0],
        )
    )

    best_val_loss = min(
        history_data.get(
            "val_loss",
            [0],
        )
    )

    metadata = {

        "model":
            "DenseNet121",

        "dataset_version":
            dataset_version,

        "image_size":
            list(IMAGE_SIZE),

        "batch_size":
            BATCH_SIZE,

        "epochs":
            EPOCHS,

        "learning_rate":
            LEARNING_RATE,

        "classes":
            CLASS_NAMES,

        "split":
            {
                "train": 0.70,
                "validation": 0.15,
                "test": 0.15,
            },

        "image_counts":
            {
                "train":
                    train_count,

                "validation":
                    validation_count,

                "test":
                    test_count,
            },

        "best_val_accuracy":
            best_val_accuracy,

        "best_val_loss":
            best_val_loss,

        "created_at":
            datetime.now().isoformat(),

        "model_path":
            str(model_path),

        "history":
            history_data,
    }

    metadata_path = (
        model_path.parent
        / (
            model_path.stem
            + "_metadata.json"
        )
    )

    with open(
        metadata_path,
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            metadata,
            file,
            indent=4,
        )

    print()
    print(
        f"Metadata sauvegardées : "
        f"{metadata_path}"
    )

In [ ]:
# ============================================================
# MAIN
# ============================================================

def main():

    print()
    print("=" * 60)
    print("      MLOps - DenseNet-121 Training")
    print("=" * 60)

    # --------------------------------------------------------
    # Dataset
    # --------------------------------------------------------

    dataset_version, dataset_dir = (
        get_latest_dataset()
    )

    print(
        f"\nDataset sélectionné : "
        f"{dataset_version}"
    )

    print(
        f"Emplacement : "
        f"{dataset_dir}"
    )

    # --------------------------------------------------------
    # Vérification
    # --------------------------------------------------------

    verify_dataset(
        dataset_dir
    )

    # --------------------------------------------------------
    # Création des 3 splits
    # --------------------------------------------------------

    (
        train_dataset,
        validation_dataset,
        test_dataset,
    ) = create_datasets(
        dataset_dir
    )

    # --------------------------------------------------------
    # Compter les images
    # --------------------------------------------------------

    print(
        "\nComptage des images..."
    )

    train_count = count_dataset_images(
        train_dataset
    )

    validation_count = count_dataset_images(
        validation_dataset
    )

    test_count = count_dataset_images(
        test_dataset
    )

    print(
        f"Train      : {train_count}"
    )

    print(
        f"Validation : {validation_count}"
    )

    print(
        f"Test       : {test_count}"
    )

    # --------------------------------------------------------
    # Optimisation
    # --------------------------------------------------------

    (
        train_dataset,
        validation_dataset,
        test_dataset,
    ) = optimize_datasets(
        train_dataset,
        validation_dataset,
        test_dataset,
    )

    # --------------------------------------------------------
    # Création du modèle
    # --------------------------------------------------------

    model = create_model()

    model.summary()

    # --------------------------------------------------------
    # Nom du modèle
    # --------------------------------------------------------

    model_path = (
        MODELS_DIR
        / f"densenet121_{dataset_version}.keras"
    )

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    with mlflow.start_run(run_name=f"train_{dataset_version}") as run:

        print()
        print(f"MLflow Run ID : {run.info.run_id}")

        mlflow.log_params(
            {
                "model": "DenseNet121",
                "dataset_version": dataset_version,
                "image_width": IMAGE_SIZE[0],
                "image_height": IMAGE_SIZE[1],
                "batch_size": BATCH_SIZE,
                "epochs": EPOCHS,
                "learning_rate": LEARNING_RATE,
                "seed": SEED,
                "train_split": TRAIN_SIZE,
                "validation_split": VALIDATION_SIZE,
                "test_split": TEST_SIZE,
                "num_classes": len(CLASS_NAMES),
                "train_images": train_count,
                "validation_images": validation_count,
                "test_images": test_count,
            }
        )

        mlflow.set_tags(
            {
                "stage": "training",
                "model_type": "DenseNet121",
                "dataset_version": dataset_version,
            }
        )

        history = train_model(
            model,
            train_dataset,
            validation_dataset,
            model_path,
        )

        history_data = history.history

        if history_data.get("accuracy"):
            mlflow.log_metric("final_train_accuracy", float(history_data["accuracy"][-1]))
        if history_data.get("loss"):
            mlflow.log_metric("final_train_loss", float(history_data["loss"][-1]))
        if history_data.get("val_accuracy"):
            mlflow.log_metric("final_val_accuracy", float(history_data["val_accuracy"][-1]))
            mlflow.log_metric("best_val_accuracy", float(max(history_data["val_accuracy"])))
        if history_data.get("val_loss"):
            mlflow.log_metric("final_val_loss", float(history_data["val_loss"][-1]))
            mlflow.log_metric("best_val_loss", float(min(history_data["val_loss"])))

        mlflow.log_artifact(str(model_path), artifact_path="model")

        # --------------------------------------------------------
        # Métadonnées
        # --------------------------------------------------------

        save_training_metadata(
            dataset_version,
            history,
            model_path,
            train_count,
            validation_count,
            test_count,
        )

        metadata_path = model_path.parent / (model_path.stem + "_metadata.json")
        if metadata_path.exists():
            mlflow.log_artifact(str(metadata_path), artifact_path="metadata")

    # --------------------------------------------------------
    # Résultat
    # --------------------------------------------------------

    print()
    print("=" * 60)
    print("Entraînement terminé")
    print("=" * 60)

    print(
        f"Modèle : {model_path}"
    )

    print(
        f"Dataset : {dataset_version}"
    )

    print(
        f"Train : {train_count}"
    )

    print(
        f"Validation : {validation_count}"
    )

    print(
        f"Test : {test_count}"
    )

    print()

In [ ]:
# ============================================================
# EXECUTION
# ============================================================

if __name__ == "__main__":
    main()